**Step 1. Install and import data from ChEMBL**

In [10]:
!pip install -q chembl_webresource_client rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 18.9 MB/s eta 0:00:00


**Step 2. Import packages**

In [11]:
import pandas as pd
import numpy as np

from chembl_webresource_client.new_client import new_client

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

**Step 3.  Check the DPP4 target**

In [76]:
TARGET_ID = "CHEMBL284"   # Human DPP4, single protein

target = new_client.target.get(TARGET_ID)

print("Target:", target["pref_name"])
print("Organism:", target["organism"])
print("Target type:", target["target_type"])

Target: Dipeptidyl peptidase 4
Organism: Homo sapiens
Target type: SINGLE PROTEIN


**Step 4. Check available DPP4 activity records**

In [77]:
activity = new_client.activity

#all records of DPP4

dpp4_all_records = activity.filter(
    target_chembl_id=TARGET_ID
).only([
    "activity_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_relation",
    "relation",
    "standard_value",
    "standard_units",
    "target_organism"
])

df_dpp4_counts = pd.DataFrame.from_records(dpp4_all_records)

print("Total DPP4 activity records:", df_dpp4_counts.shape[0])
print("Unique DPP4 compounds with any activity:", df_dpp4_counts["molecule_chembl_id"].nunique())

# IC50 only
df_dpp4_ic50_preview = df_dpp4_counts[
    df_dpp4_counts["standard_type"] == "IC50"
].copy()

print("Total DPP4 IC50 records:", df_dpp4_ic50_preview.shape[0])
print("Unique DPP4 compounds with IC50:", df_dpp4_ic50_preview["molecule_chembl_id"].nunique())

Total DPP4 activity records: 8398
Unique DPP4 compounds with any activity: 5752
Total DPP4 IC50 records: 5819
Unique DPP4 compounds with IC50: 4834


**Step 5. Extract IC50 activity data directly from ChEMBL**

**5-1) Retrieve raw ChEMBL records**

In [102]:
activity = new_client.activity

records = activity.filter(
    target_chembl_id=TARGET_ID,
    standard_type="IC50"
)

df_raw = pd.DataFrame.from_records(records)

print("Raw IC50 records:", df_raw.shape)
df_raw.head()

Raw IC50 records: (5819, 47)


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,105501,[],CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,None,None,BAO_0000190,...,Homo sapiens,Dipeptidyl peptidase 4,9606,None,None,IC50,uM,UO_0000065,None,217.0
1,None,None,105501,[],CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,None,None,BAO_0000190,...,Homo sapiens,Dipeptidyl peptidase 4,9606,None,None,IC50,uM,UO_0000065,None,217.0
2,None,None,106644,[],CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,None,None,BAO_0000190,...,Homo sapiens,Dipeptidyl peptidase 4,9606,None,None,IC50,uM,UO_0000065,None,41.0
3,None,None,106647,[],CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,None,None,BAO_0000190,...,Homo sapiens,Dipeptidyl peptidase 4,9606,None,None,IC50,uM,UO_0000065,None,15.0
4,None,None,106650,[],CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,None,None,BAO_0000190,...,Homo sapiens,Dipeptidyl peptidase 4,9606,None,None,IC50,uM,UO_0000065,None,500.0


**5-2) Remove exact duplicated rows**

In [103]:
print("Available columns:")
print(df_raw.columns.tolist())

# Remove exact duplicated activity rows
before = df_raw.shape[0]

duplicate_subset = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation"
]

duplicate_subset = [c for c in duplicate_subset if c in df_raw.columns]

df_raw = df_raw.drop_duplicates(subset=duplicate_subset).copy()

after = df_raw.shape[0]

print("Raw records before duplicate-row removal:", before)
print("Raw records after duplicate-row removal:", after)
print("Exact duplicated rows removed:", before - after)

Available columns:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value']
Raw records before duplicate-row removal: 5819
Raw records after duplicate-row removal: 5818
Exact duplicated rows removed: 1


**5-3) Save deduplicated raw IC50 records**

In [104]:
df_raw.to_csv("DPP4_ChEMBL_raw_IC50_deduplicated.csv", index=False)
print("Saved: DPP4_ChEMBL_raw_IC50_deduplicated.csv")

Saved: DPP4_ChEMBL_raw_IC50_deduplicated.csv


**5-4) Keep important columns only**

In [106]:
cols_to_keep = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "target_chembl_id",
    "target_organism",
    "standard_type",
    "standard_relation",
    "relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "data_validity_comment",
    "potential_duplicate",
    "standard_flag",
    "document_chembl_id",
    "src_id"
]

cols_existing = [c for c in cols_to_keep if c in df_raw.columns]
df = df_raw[cols_existing].copy()

print(df.shape)
df.head()

(5818, 18)


,activity_id,molecule_chembl_id,assay_chembl_id,assay_description,assay_type,target_chembl_id,target_organism,standard_type,standard_relation,relation,standard_value,standard_units,pchembl_value,data_validity_comment,potential_duplicate,standard_flag,document_chembl_id,src_id
0,105501,CHEMBL93558,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,217000.0,nM,None,Outside typical range,0,1,CHEMBL1135386,1
2,106644,CHEMBL443622,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,41000.0,nM,4.39,None,0,1,CHEMBL1135386,1
3,106647,CHEMBL403882,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,15000.0,nM,4.82,None,0,1,CHEMBL1135386,1
4,106650,CHEMBL328655,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,>,>,500000.0,nM,None,Outside typical range,0,1,CHEMBL1135386,1
5,108924,CHEMBL328795,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,188000.0,nM,None,Outside typical range,0,1,CHEMBL1135386,1


**5-5) Convert numeric columns**

In [107]:
for col in ["standard_value", "pchembl_value"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

**Step 6. Strict IC50 filtering**

In [108]:
relation_col = "standard_relation" if "standard_relation" in df.columns else "relation"

mask = (
    (df["standard_type"] == "IC50") &
    (df["standard_units"] == "nM") &
    (df[relation_col] == "=") &
    (df["standard_value"].notna()) &
    (df["standard_value"] > 0)
)

if "target_organism" in df.columns:
    mask &= df["target_organism"] == "Homo sapiens"

if "data_validity_comment" in df.columns:
    mask &= df["data_validity_comment"].isna()

if "potential_duplicate" in df.columns:
    mask &= ~df["potential_duplicate"].astype(str).str.upper().isin(["1", "TRUE"])

if "standard_flag" in df.columns:
    mask &= df["standard_flag"].astype(str).str.upper().isin(["1", "TRUE"])

df_ic50 = df[mask].copy()

df_ic50 = df_ic50.rename(columns={"standard_value": "IC50_nM"})

df_ic50["pIC50"] = 9 - np.log10(df_ic50["IC50_nM"])

print("After strict IC50 filtering:", df_ic50.shape)
print("Unique compounds after strict filtering:", df_ic50["molecule_chembl_id"].nunique())

df_ic50.head()

After strict IC50 filtering: (4278, 19)
Unique compounds after strict filtering: 3924


,activity_id,molecule_chembl_id,assay_chembl_id,assay_description,assay_type,target_chembl_id,target_organism,standard_type,standard_relation,relation,IC50_nM,standard_units,pchembl_value,data_validity_comment,potential_duplicate,standard_flag,document_chembl_id,src_id,pIC50
2,106644,CHEMBL443622,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,41000.0,nM,4.39,None,0,1,CHEMBL1135386,1,4.387216
3,106647,CHEMBL403882,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,15000.0,nM,4.82,None,0,1,CHEMBL1135386,1,4.823909
7,111673,CHEMBL317126,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,14000.0,nM,4.85,None,0,1,CHEMBL1135386,1,4.853872
8,113099,CHEMBL383705,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,4000.0,nM,5.40,None,0,1,CHEMBL1135386,1,5.397940
13,121780,CHEMBL98008,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,17000.0,nM,4.77,None,0,1,CHEMBL1135386,1,4.769551


**Save the file**

In [109]:
df_ic50.to_csv(
    "DPP4_ChEMBL_IC50_strict_filtered.csv",
    index=False
)

print("DPP4_ChEMBL_IC50_strict_filtered.csv is available")

DPP4_ChEMBL_IC50_strict_filtered.csv is available


**Step 7. Retrieve molecule structures and molecular properties from ChEMBL**

In [110]:
molecule = new_client.molecule

unique_mols = df_ic50["molecule_chembl_id"].dropna().unique().tolist()

print("Unique compounds for molecule retrieval:", len(unique_mols))

def chunk_list(lst, n=100):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

mol_records = []

for batch in chunk_list(unique_mols, 100):
    records = molecule.filter(
        molecule_chembl_id__in=batch
    ).only([
        "molecule_chembl_id",
        "pref_name",
        "molecule_structures",
        "molecule_properties"
    ])

    mol_records.extend(list(records))

df_mol = pd.DataFrame(mol_records)

print("Retrieved molecule records:", df_mol.shape)
df_mol.head()

Unique compounds for molecule retrieval: 3924
Retrieved molecule records: (3924, 4)


,molecule_chembl_id,molecule_properties,molecule_structures,pref_name
0,CHEMBL9512,"{'alogp': '1.83', 'aromatic_rings': 2, 'full_m...",{'canonical_smiles': 'Cl.NCC(=O)Nc1ccc(OP(=O)(...,None
1,CHEMBL269757,"{'alogp': '4.78', 'aromatic_rings': 4, 'full_m...",{'canonical_smiles': 'Cl.O=C(CNC(=O)c1ccccc1)N...,None
2,CHEMBL430281,"{'alogp': '2.35', 'aromatic_rings': 1, 'full_m...",{'canonical_smiles': 'Cl.NC(=O)Nc1cccc(OP(=O)(...,None
3,CHEMBL265344,"{'alogp': '3.95', 'aromatic_rings': 2, 'full_m...",{'canonical_smiles': 'CC(=O)Nc1ccc(OP(=O)(Oc2c...,None
4,CHEMBL273851,"{'alogp': '4.04', 'aromatic_rings': 2, 'full_m...",{'canonical_smiles': 'Cl.O=C([C@H]1CCCN1)N1CCC...,None


**Step 8. Extract canonical SMILES and molecular properties**

In [111]:
def get_canonical_smiles(x):
    if isinstance(x, dict):
        return x.get("canonical_smiles")
    return None

def get_prop(x, key):
    if isinstance(x, dict):
        return x.get(key)
    return None

df_mol["canonical_smiles"] = df_mol["molecule_structures"].apply(get_canonical_smiles)

property_cols = [
    "mw_freebase",
    "alogp",
    "hba",
    "hbd",
    "psa",
    "rtb",
    "num_ro5_violations"
]

for col in property_cols:
    df_mol[col] = df_mol["molecule_properties"].apply(lambda x: get_prop(x, col))
    df_mol[col] = pd.to_numeric(df_mol[col], errors="coerce")

df_mol_clean = df_mol[
    ["molecule_chembl_id", "pref_name", "canonical_smiles"] + property_cols
].copy()

df_ic50 = df_ic50.merge(
    df_mol_clean,
    on="molecule_chembl_id",
    how="left"
)

df_ic50 = df_ic50.dropna(subset=["canonical_smiles"]).copy()

print("Strict IC50 records with SMILES:", df_ic50.shape)
print("Missing canonical SMILES:", df_ic50["canonical_smiles"].isna().sum())
print("Unique compounds with SMILES:", df_ic50["molecule_chembl_id"].nunique())

df_ic50.head()

Strict IC50 records with SMILES: (4274, 28)
Missing canonical SMILES: 0
Unique compounds with SMILES: 3920


,activity_id,molecule_chembl_id,assay_chembl_id,assay_description,assay_type,target_chembl_id,target_organism,standard_type,standard_relation,relation,...,pIC50,pref_name,canonical_smiles,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations
0,106644,CHEMBL443622,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,...,4.387216,None,C[C@H](N)C(=O)N1CCCC1,142.20,-0.04,2.0,1.0,46.33,1.0,0.0
1,106647,CHEMBL403882,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,...,4.823909,None,O=C([C@@H]1CCCN1)N1CCCC1,168.24,0.36,2.0,1.0,32.34,1.0,0.0
2,111673,CHEMBL317126,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,...,4.853872,None,NC(Cc1ccc(O)cc1)C(=O)N1CCCC1,234.30,0.88,3.0,2.0,66.56,3.0,0.0
3,113099,CHEMBL383705,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,...,5.397940,None,CC(C)[C@H](N)C(=O)N1CCCC1,170.26,0.59,2.0,1.0,46.33,2.0,0.0
4,121780,CHEMBL98008,CHEMBL666573,In vitro inhibition of human Dipeptidylpeptida...,B,CHEMBL284,Homo sapiens,IC50,=,=,...,4.769551,None,N[C@@H](CC1CCCCC1)C(=O)N1CCCC1,224.35,1.91,2.0,1.0,46.33,3.0,0.0


**Step 9. Check assay types before duplicate handling**

In [112]:
print("Assay type distribution in strict IC50 dataset:")
print(df_ic50["assay_type"].value_counts(dropna=False))

df_non_b = df_ic50[df_ic50["assay_type"] != "B"].copy()

print("\nNon-biochemical strict IC50 records:", df_non_b.shape)

df_non_b[
    [
        "molecule_chembl_id",
        "pref_name",
        "canonical_smiles",
        "assay_type",
        "IC50_nM",
        "pIC50",
        "assay_chembl_id",
        "document_chembl_id"
    ]
].head(20)

Assay type distribution in strict IC50 dataset:
assay_type
B    4238
F      36
Name: count, dtype: int64

Non-biochemical strict IC50 records: (36, 28)


,molecule_chembl_id,pref_name,canonical_smiles,assay_type,IC50_nM,pIC50,assay_chembl_id,document_chembl_id
179,CHEMBL289562,None,NCc1c(N)nc(-c2ccccc2)nc1-c1ccccc1,F,42000.0,4.376751,CHEMBL664533,CHEMBL1147631
181,CHEMBL37229,None,COc1ccccc1-c1nc(-c2ccccc2)nc(N)c1CN,F,1500.0,5.823909,CHEMBL664533,CHEMBL1147631
182,CHEMBL37536,None,NCc1c(N)nc(-c2ccccc2F)nc1-c1ccc(Cl)cc1Cl,F,47.0,7.327902,CHEMBL664533,CHEMBL1147631
183,CHEMBL37654,None,NCc1c(N)nc(-c2cccc(C(F)(F)F)c2)nc1-c1ccc(Cl)cc1Cl,F,130.0,6.886057,CHEMBL664533,CHEMBL1147631
185,CHEMBL34764,None,NCc1c(N)nc(-c2ccc(C(F)(F)F)cc2)nc1-c1ccc(Cl)cc1Cl,F,180.0,6.744727,CHEMBL664533,CHEMBL1147631
186,CHEMBL287534,None,Cc1cccc(-c2nc(-c3ccccc3)nc(N)c2CN)c1,F,20000.0,4.698970,CHEMBL664533,CHEMBL1147631
187,CHEMBL34603,None,NCc1c(N)nc(-c2ccccc2)nc1-c1ccc(Cl)cc1,F,1400.0,5.853872,CHEMBL664533,CHEMBL1147631
188,CHEMBL36333,None,NCc1c(N)nc(-c2ccccc2)nc1-c1ccccc1Cl,F,2500.0,5.602060,CHEMBL664533,CHEMBL1147631
189,CHEMBL35451,None,NCc1c(N)nc(-c2ccccc2)nc1-c1cccc(F)c1,F,40000.0,4.397940,CHEMBL664533,CHEMBL1147631
191,CHEMBL38165,None,Cc1ccc(-c2nc(-c3ccccc3)nc(N)c2CN)cc1,F,1000.0,6.000000,CHEMBL664533,CHEMBL1147631


* Binding assay : B assay
* Functional assay : F assay

**Step 10. Choose source dataset**

In [113]:
USE_B_ONLY = True   # recommended for cleaner QSAR

if USE_B_ONLY:
    source_df = df_ic50[df_ic50["assay_type"] == "B"].copy()
    print("Using B assay records only.")
else:
    source_df = df_ic50.copy()
    print("Using all strict IC50 records.")

print("Source records:", source_df.shape)
print("Unique ChEMBL compounds:", source_df["molecule_chembl_id"].nunique())

Using B assay records only.
Source records: (4238, 28)
Unique ChEMBL compounds: 3885


**Step 11. Remove salts/counterions and create parent SMILES**

In [114]:
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

# Suppress RDKit info messages
RDLogger.DisableLog("rdApp.*")

largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()

def get_parent_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    # Keep largest molecular fragment
    try:
        mol = largest_fragment_chooser.choose(mol)
    except:
        fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        if len(fragments) == 0:
            return None
        mol = max(fragments, key=lambda m: m.GetNumHeavyAtoms())

    # Neutralize charges when possible
    try:
        mol = uncharger.uncharge(mol)
    except:
        pass

    try:
        Chem.SanitizeMol(mol)
    except:
        return None

    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

source_df["has_multiple_fragments_raw"] = (
    source_df["canonical_smiles"].astype(str).str.contains(r"\.")
)

source_df["parent_smiles"] = source_df["canonical_smiles"].apply(get_parent_smiles)

source_df = source_df.dropna(subset=["parent_smiles"]).copy()

source_df["has_multiple_fragments_parent"] = (
    source_df["parent_smiles"].astype(str).str.contains(r"\.")
)

print("Raw multi-fragment SMILES:", source_df["has_multiple_fragments_raw"].sum())
print("Parent multi-fragment SMILES:", source_df["has_multiple_fragments_parent"].sum())
print("Records after parent SMILES cleanup:", source_df.shape)
print("Unique parent SMILES:", source_df["parent_smiles"].nunique())

source_df[
    ["molecule_chembl_id", "canonical_smiles", "parent_smiles", "pIC50"]
].head()

Raw multi-fragment SMILES: 831
Parent multi-fragment SMILES: 0
Records after parent SMILES cleanup: (4238, 31)
Unique parent SMILES: 3850


,molecule_chembl_id,canonical_smiles,parent_smiles,pIC50
0,CHEMBL443622,C[C@H](N)C(=O)N1CCCC1,C[C@H](N)C(=O)N1CCCC1,4.387216
1,CHEMBL403882,O=C([C@@H]1CCCN1)N1CCCC1,O=C([C@@H]1CCCN1)N1CCCC1,4.823909
2,CHEMBL317126,NC(Cc1ccc(O)cc1)C(=O)N1CCCC1,NC(Cc1ccc(O)cc1)C(=O)N1CCCC1,4.853872
3,CHEMBL383705,CC(C)[C@H](N)C(=O)N1CCCC1,CC(C)[C@H](N)C(=O)N1CCCC1,5.397940
4,CHEMBL98008,N[C@@H](CC1CCCCC1)C(=O)N1CCCC1,N[C@@H](CC1CCCCC1)C(=O)N1CCCC1,4.769551


**Check the list of removed salts**

In [115]:
from rdkit import Chem

def get_fragments_as_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
    frag_smiles = [
        Chem.MolToSmiles(frag, canonical=True, isomericSmiles=True)
        for frag in fragments
    ]

    return ".".join(frag_smiles)

def get_removed_fragments(smiles):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)

    if len(fragments) <= 1:
        return ""

    # Identify largest fragment by heavy atom count
    largest_idx = max(
        range(len(fragments)),
        key=lambda i: fragments[i].GetNumHeavyAtoms()
    )

    removed = []
    for i, frag in enumerate(fragments):
        if i != largest_idx:
            removed.append(
                Chem.MolToSmiles(frag, canonical=True, isomericSmiles=True)
            )

    return ".".join(removed)

# Compounds whose raw SMILES had multiple fragments
salt_removed_df = source_df[
    source_df["has_multiple_fragments_raw"] == True
].copy()

salt_removed_df["raw_fragments"] = salt_removed_df["canonical_smiles"].apply(get_fragments_as_smiles)
salt_removed_df["removed_fragments"] = salt_removed_df["canonical_smiles"].apply(get_removed_fragments)
salt_removed_df["parent_changed"] = (
    salt_removed_df["canonical_smiles"] != salt_removed_df["parent_smiles"]
)

print("Records with raw multi-fragment SMILES:", salt_removed_df.shape[0])
print("Records where parent SMILES changed:", salt_removed_df["parent_changed"].sum())
print("Parent multi-fragment SMILES remaining:", source_df["has_multiple_fragments_parent"].sum())

cols_to_show = [
    "molecule_chembl_id",
    "pref_name",
    "canonical_smiles",
    "parent_smiles",
    "removed_fragments",
    "pIC50"
]

available_cols = [col for col in cols_to_show if col in salt_removed_df.columns]

salt_removed_df[available_cols].head(20)

Records with raw multi-fragment SMILES: 831
Records where parent SMILES changed: 831
Parent multi-fragment SMILES remaining: 0


,molecule_chembl_id,pref_name,canonical_smiles,parent_smiles,removed_fragments,pIC50
28,CHEMBL265344,None,CC(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(C)=O)cc2)C2CCCN2...,CC(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(C)=O)cc2)C2CCCN2...,Cl,6.397940
29,CHEMBL9237,None,COP(=O)(Oc1ccccc1-c1ccccc1O)C1CCCN1C(=O)[C@@H]...,COP(=O)(Oc1ccccc1-c1ccccc1O)C1CCCN1C(=O)[C@@H]...,Cl,4.327902
30,CHEMBL9724,None,Cl.O=C([C@@H]1CCCN1)N1CCCC1P1(=O)Oc2ccccc2-c2c...,O=C([C@@H]1CCCN1)N1CCCC1P1(=O)Oc2ccccc2-c2cccc...,Cl,4.508638
31,CHEMBL9561,None,COc1ccc(OP(=O)(Oc2ccc(OC)cc2)C2CCCN2C(=O)C2CCC...,COc1ccc(OP(=O)(Oc2ccc(OC)cc2)C2CCCN2C(=O)C2CCC...,Cl,4.657577
32,CHEMBL9238,None,COC(=O)CCNC(=O)c1ccc(OP(=O)(Oc2ccc(C(=O)NCCC(=...,COC(=O)CCNC(=O)c1ccc(OP(=O)(Oc2ccc(C(=O)NCCC(=...,Cl,7.443697
33,CHEMBL9512,None,Cl.NCC(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(=O)CN)cc2)C2...,NCC(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(=O)CN)cc2)C2CCC...,Cl,6.301030
34,CHEMBL269209,None,C[C@H](N)C(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(=O)[C@@H...,C[C@H](N)C(=O)Nc1ccc(OP(=O)(Oc2ccc(NC(=O)[C@@H...,Cl,6.221849
35,CHEMBL273851,None,Cl.O=C([C@H]1CCCN1)N1CCCC1P(=O)(Oc1ccccc1)Oc1c...,O=C([C@H]1CCCN1)N1CCCC1P(=O)(Oc1ccccc1)Oc1ccccc1,Cl,5.301030
36,CHEMBL9295,None,CCCNC(=O)c1ccc(OP(=O)(Oc2ccc(C(=O)NCCC)cc2)C2C...,CCCNC(=O)c1ccc(OP(=O)(Oc2ccc(C(=O)NCCC)cc2)C2C...,Cl,7.522879
37,CHEMBL269757,None,Cl.O=C(CNC(=O)c1ccccc1)Nc1ccc(OP(=O)(Oc2ccc(NC...,O=C(CNC(=O)c1ccccc1)Nc1ccc(OP(=O)(Oc2ccc(NC(=O...,Cl,6.154902


**Step 12. Handle duplicate compounds by parent SMILES**

In [116]:
def join_unique(x):
    return ";".join(sorted(x.dropna().astype(str).unique()))

# Make sure numeric columns are numeric
for col in ["IC50_nM", "pIC50", "pchembl_value"]:
    if col in source_df.columns:
        source_df[col] = pd.to_numeric(source_df[col], errors="coerce")

df_unique = (
    source_df
    .groupby("parent_smiles", as_index=False)
    .agg(
        molecule_chembl_id=("molecule_chembl_id", join_unique),
        pref_name=("pref_name", lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else np.nan),
        canonical_smiles_examples=("canonical_smiles", join_unique),
        n_records=("pIC50", "size"),

        IC50_nM_median=("IC50_nM", "median"),
        IC50_nM_min=("IC50_nM", "min"),
        IC50_nM_max=("IC50_nM", "max"),

        pIC50_median=("pIC50", "median"),
        pIC50_min=("pIC50", "min"),
        pIC50_max=("pIC50", "max"),
        pIC50_std=("pIC50", "std"),

        pchembl_median=("pchembl_value", "median"),

        assay_type=("assay_type", join_unique),
        assay_chembl_id=("assay_chembl_id", join_unique),
        document_chembl_id=("document_chembl_id", join_unique),

        mw_freebase=("mw_freebase", "median"),
        alogp=("alogp", "median"),
        hba=("hba", "median"),
        hbd=("hbd", "median"),
        psa=("psa", "median"),
        rtb=("rtb", "median"),
        num_ro5_violations=("num_ro5_violations", "median")
    )
)

df_unique["pIC50_std"] = df_unique["pIC50_std"].fillna(0)
df_unique["pIC50_range"] = df_unique["pIC50_max"] - df_unique["pIC50_min"]

print("Source dataset used:", "B assay only" if USE_B_ONLY else "All strict IC50 records")
print("Records before duplicate handling:", source_df.shape)
print("Unique ChEMBL compound IDs before duplicate handling:", source_df["molecule_chembl_id"].nunique())
print("Unique parent compounds before duplicate-consistency filter:", df_unique.shape)

df_unique.head()

Source dataset used: B assay only
Records before duplicate handling: (4238, 31)
Unique ChEMBL compound IDs before duplicate handling: 3885
Unique parent compounds before duplicate-consistency filter: (3850, 24)


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,IC50_nM_median,IC50_nM_min,IC50_nM_max,pIC50_median,pIC50_min,...,assay_chembl_id,document_chembl_id,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations,pIC50_range
0,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,1,28050.00,28050.00,28050.00,4.552067,4.552067,...,CHEMBL4232372,CHEMBL4229387,318.13,2.62,6.0,1.0,68.24,1.0,0.0,0.000000
1,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,1,60.00,60.00,60.00,7.221849,7.221849,...,CHEMBL939125,CHEMBL1143890,276.34,-0.57,4.0,1.0,76.44,4.0,0.0,0.000000
2,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,1,16.00,16.00,16.00,7.795880,7.795880,...,CHEMBL3888356,CHEMBL3886676,417.47,0.38,9.0,1.0,114.87,4.0,0.0,0.000000
3,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,1,1.33,1.33,1.33,8.876148,8.876148,...,CHEMBL5247172,CHEMBL5244277,458.53,0.76,10.0,1.0,116.86,4.0,0.0,0.000000
4,C#C[C@@H]1C[C@H](F)CN1C(=O)CC(C)(C)CC(=O)N[C@H...,CHEMBL1082996,NaN,C#C[C@@H]1C[C@H](F)CN1C(=O)CC(C)(C)CC(=O)N[C@H...,3,210.00,15.00,847.00,6.677781,6.072117,...,CHEMBL1119677;CHEMBL1119686;CHEMBL1119687,CHEMBL1158599,404.91,3.69,2.0,1.0,49.41,6.0,0.0,1.751792


**Check the parent-SMILES duplicate handling list**

In [117]:
print("Missing parent_smiles:", df_unique["parent_smiles"].isna().sum())
print("Duplicated parent_smiles:", df_unique["parent_smiles"].duplicated().sum())
print("Multi-fragment parent_smiles:", df_unique["parent_smiles"].astype(str).str.contains(r"\.").sum())

print("\nNumber of parent compounds with duplicate records:")
print((df_unique["n_records"] > 1).sum())

print("\nTop duplicated parent compounds:")
df_unique.sort_values("n_records", ascending=False)[
    [
        "parent_smiles",
        "molecule_chembl_id",
        "pref_name",
        "n_records",
        "pIC50_median",
        "pIC50_min",
        "pIC50_max",
        "pIC50_range",
        "assay_chembl_id"
    ]
].head(20)

Missing parent_smiles: 0
Duplicated parent_smiles: 0
Multi-fragment parent_smiles: 0

Number of parent compounds with duplicate records:
250

Top duplicated parent compounds:


,parent_smiles,molecule_chembl_id,pref_name,n_records,pIC50_median,pIC50_min,pIC50_max,pIC50_range,assay_chembl_id
3083,N[C@@H](CC(=O)N1CCn2c(nnc2C(F)(F)F)C1)Cc1cc(F)...,CHEMBL1201174;CHEMBL1422;CHEMBL393336;CHEMBL53...,SITAGLIPTIN,30,7.744727,6.920819,9.060481,2.139662,CHEMBL1011679;CHEMBL1034787;CHEMBL1119687;CHEM...
2254,N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,CHEMBL142703,VILDAGLIPTIN,18,7.668621,4.619789,9.236572,4.616783,CHEMBL1030770;CHEMBL1057246;CHEMBL1061470;CHEM...
2135,Cn1c(=O)cc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,CHEMBL376359,ALOGLIPTIN,16,8.374303,7.619789,9.000000,1.380211,CHEMBL1663348;CHEMBL1664314;CHEMBL1821018;CHEM...
628,CC(C)[C@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL67279,TALABOSTAT,10,8.213564,5.920819,10.000000,4.079181,CHEMBL1768625;CHEMBL1768626;CHEMBL2026783;CHEM...
1694,C[C@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL63860,NaN,10,7.062469,4.455932,9.585027,5.129095,CHEMBL1768625;CHEMBL1768626;CHEMBL2066843;CHEM...
229,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nc(C)...,CHEMBL237500,LINAGLIPTIN,8,9.198970,7.813326,10.000000,2.186674,CHEMBL1954392;CHEMBL2389862;CHEMBL4379983;CHEM...
1058,CN(C)C(=O)[C@@H](c1ccc(-c2ccc(F)cc2)cc1)[C@H](...,CHEMBL191067;CHEMBL237119,NaN,7,7.744727,6.412289,7.920819,1.508530,CHEMBL827856;CHEMBL828036;CHEMBL828037;CHEMBL8...
1538,CS(=O)(=O)n1cc2c(n1)CN([C@H]1CO[C@H](c3cc(F)cc...,CHEMBL2105762,OMARIGLIPTIN,6,8.619337,8.376751,8.795880,0.419129,CHEMBL3243518;CHEMBL3737789;CHEMBL3862754;CHEM...
2434,N#C[C@@H]1C[C@@H]2C[C@@H]2N1C(=O)[C@@H](N)C12C...,CHEMBL385517,SAXAGLIPTIN ANHYDROUS,6,8.495700,7.301030,9.221849,1.920819,CHEMBL1061470;CHEMBL1119677;CHEMBL1816914;CHEM...
1065,CN(C)C(=O)[C@@H](c1ccc(-c2ccc3ncnn3c2)cc1)[C@H...,CHEMBL237337;CHEMBL377221,NaN,5,8.366532,7.356547,8.397940,1.041393,CHEMBL3074800;CHEMBL866326;CHEMBL866327;CHEMBL...


**Step 13. Remove inconsistent duplicates**

In [118]:
DUPLICATE_PIC50_RANGE_CUTOFF = 0.2

df_curated = df_unique[
    (df_unique["n_records"] == 1) |
    (df_unique["pIC50_range"] <= DUPLICATE_PIC50_RANGE_CUTOFF)
].copy()

print("Curated unique parent compounds:", df_curated.shape)
print("Removed inconsistent parent duplicates:", df_unique.shape[0] - df_curated.shape[0])

df_curated.head()

Curated unique parent compounds: (3666, 24)
Removed inconsistent parent duplicates: 184


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,IC50_nM_median,IC50_nM_min,IC50_nM_max,pIC50_median,pIC50_min,...,assay_chembl_id,document_chembl_id,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations,pIC50_range
0,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,1,28050.00,28050.00,28050.00,4.552067,4.552067,...,CHEMBL4232372,CHEMBL4229387,318.13,2.62,6.0,1.0,68.24,1.0,0.0,0.0
1,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,1,60.00,60.00,60.00,7.221849,7.221849,...,CHEMBL939125,CHEMBL1143890,276.34,-0.57,4.0,1.0,76.44,4.0,0.0,0.0
2,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,1,16.00,16.00,16.00,7.795880,7.795880,...,CHEMBL3888356,CHEMBL3886676,417.47,0.38,9.0,1.0,114.87,4.0,0.0,0.0
3,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,1,1.33,1.33,1.33,8.876148,8.876148,...,CHEMBL5247172,CHEMBL5244277,458.53,0.76,10.0,1.0,116.86,4.0,0.0,0.0
5,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,CHEMBL1650422,NaN,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,1,5.00,5.00,5.00,8.301030,8.301030,...,CHEMBL1664314,CHEMBL1649419,333.40,1.07,6.0,1.0,87.94,3.0,0.0,0.0


**Check the compounds removed due to inconsistent duplicate pIC50**

In [120]:
removed_inconsistent = df_unique.loc[
    ~df_unique.index.isin(df_curated.index)
].copy()

print(
    "Compounds removed due to inconsistent duplicate pIC50:",
    removed_inconsistent.shape[0]
)

removed_cols = [
    "parent_smiles",
    "molecule_chembl_id",
    "pref_name",
    "canonical_smiles_examples",
    "n_records",
    "pIC50_median",
    "pIC50_min",
    "pIC50_max",
    "pIC50_range",
    "assay_type",
    "assay_chembl_id",
    "document_chembl_id"
]

available_cols = [col for col in removed_cols if col in removed_inconsistent.columns]

removed_inconsistent[available_cols].sort_values(
    "pIC50_range",
    ascending=False
).head(20)

Compounds removed due to inconsistent duplicate pIC50: 184


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,pIC50_median,pIC50_min,pIC50_max,pIC50_range,assay_type,assay_chembl_id,document_chembl_id
1694,C[C@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL63860,NaN,C[C@H](N)C(=O)N1CCC[C@H]1B(O)O,10,7.062469,4.455932,9.585027,5.129095,B,CHEMBL1768625;CHEMBL1768626;CHEMBL2066843;CHEM...,CHEMBL1129318;CHEMBL1139886;CHEMBL1765065;CHEM...
1701,C[C@H](N)C(=S)N1CCC[C@H]1B(O)O,CHEMBL3686733,NaN,C[C@H](N)C(=S)N1CCC[C@H]1B(O)O,3,6.698970,4.698970,9.455932,4.756962,B,CHEMBL3706075;CHEMBL5737203,CHEMBL3638744;CHEMBL5728672
2254,N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,CHEMBL142703,VILDAGLIPTIN,N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,18,7.668621,4.619789,9.236572,4.616783,B,CHEMBL1030770;CHEMBL1057246;CHEMBL1061470;CHEM...,CHEMBL1138669;CHEMBL1140240;CHEMBL1146708;CHEM...
1560,C[C@@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL196120,NaN,C[C@@H](N)C(=O)N1CCC[C@H]1B(O)O,2,6.887345,4.677781,9.096910,4.419129,B,CHEMBL880567;CHEMBL913151,CHEMBL1139886;CHEMBL1141049
3712,O=C([C@@H]1CCCN1)N1CCC[C@H]1B(O)O,CHEMBL63895,NaN,O=C([C@@H]1CCCN1)N1CCC[C@H]1B(O)O,4,6.792513,4.638272,8.958607,4.320335,B,CHEMBL3706075;CHEMBL5737203;CHEMBL665783,CHEMBL1129318;CHEMBL3638744;CHEMBL5728672
2758,NCC(=O)N1CCC[C@H]1B(O)O,CHEMBL67089,NaN,NCC(=O)N1CCC[C@H]1B(O)O,4,6.520479,4.795880,9.000000,4.204120,B,CHEMBL2346132;CHEMBL3706075;CHEMBL665783,CHEMBL1129318;CHEMBL2331146;CHEMBL3638744
628,CC(C)[C@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL67279,TALABOSTAT,CC(C)[C@H](N)C(=O)N1CCC[C@H]1B(O)O,10,8.213564,5.920819,10.000000,4.079181,B,CHEMBL1768625;CHEMBL1768626;CHEMBL2026783;CHEM...,CHEMBL1129318;CHEMBL1138385;CHEMBL1139886;CHEM...
820,CCC[C@H](N)C(=S)N1CCC[C@H]1B(O)O,CHEMBL3686746,NaN,CCC[C@H](N)C(=S)N1CCC[C@H]1B(O)O,2,6.758563,4.721246,8.795880,4.074634,B,CHEMBL3706075,CHEMBL3638744
3326,N[C@H](C(=O)N1CCC[C@H]1B(O)O)C1CCCCC1,CHEMBL3686737,NaN,N[C@H](C(=O)N1CCC[C@H]1B(O)O)C1CCCCC1,5,6.420216,5.045757,9.000000,3.954243,B,CHEMBL3706075;CHEMBL5737203,CHEMBL3638744;CHEMBL5728672
605,CC(C)[C@@H](N)C(=O)N1CCC[C@H]1B(O)O,CHEMBL305170,NaN,CC(C)[C@@H](N)C(=O)N1CCC[C@H]1B(O)O,2,7.580575,5.638272,9.522879,3.884607,B,CHEMBL880567;CHEMBL913151,CHEMBL1139886;CHEMBL1141049


**Re-check the curated parent dataset**

In [121]:
print("Missing parent_smiles:", df_curated["parent_smiles"].isna().sum())
print("Duplicated parent_smiles:", df_curated["parent_smiles"].duplicated().sum())
print("Multi-fragment parent_smiles:", df_curated["parent_smiles"].astype(str).str.contains(r"\.").sum())

Missing parent_smiles: 0
Duplicated parent_smiles: 0
Multi-fragment parent_smiles: 0


**Step 14. Add activity class**

In [122]:
df_curated["activity_class"] = pd.cut(
    df_curated["pIC50_median"],
    bins=[-np.inf, 6, 7, np.inf],
    labels=["Low", "Moderate", "High"],
    right=False
)

print("Activity class distribution:")
print(df_curated["activity_class"].value_counts())

print("\npIC50 summary:")
print(df_curated["pIC50_median"].describe())

df_curated.head()

Activity class distribution:
activity_class
High        2113
Moderate     908
Low          645
Name: count, dtype: int64

pIC50 summary:
count    3666.000000
mean        7.111067
std         1.213927
min         4.000000
25%         6.367038
50%         7.193820
75%         8.000000
max        10.096910
Name: pIC50_median, dtype: float64


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,IC50_nM_median,IC50_nM_min,IC50_nM_max,pIC50_median,pIC50_min,...,document_chembl_id,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations,pIC50_range,activity_class
0,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,1,28050.00,28050.00,28050.00,4.552067,4.552067,...,CHEMBL4229387,318.13,2.62,6.0,1.0,68.24,1.0,0.0,0.0,Low
1,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,1,60.00,60.00,60.00,7.221849,7.221849,...,CHEMBL1143890,276.34,-0.57,4.0,1.0,76.44,4.0,0.0,0.0,High
2,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,1,16.00,16.00,16.00,7.795880,7.795880,...,CHEMBL3886676,417.47,0.38,9.0,1.0,114.87,4.0,0.0,0.0,High
3,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,1,1.33,1.33,1.33,8.876148,8.876148,...,CHEMBL5244277,458.53,0.76,10.0,1.0,116.86,4.0,0.0,0.0,High
5,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,CHEMBL1650422,NaN,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,1,5.00,5.00,5.00,8.301030,8.301030,...,CHEMBL1649419,333.40,1.07,6.0,1.0,87.94,3.0,0.0,0.0,High


**Step 15. Save the files**

In [123]:
import os

# 1. Strict IC50 records after assay/source selection and parent-smiles annotation
source_df.to_csv(
    "DPP4_ChEMBL_IC50_strict_records_parent_annotated.csv",
    index=False
)

# 2. Unique parent compounds before duplicate-consistency filtering
df_unique.to_csv(
    "DPP4_ChEMBL_IC50_parent_unique_before_duplicate_filter.csv",
    index=False
)

# 3. Removed inconsistent duplicate parent compounds
removed_inconsistent.to_csv(
    "DPP4_removed_inconsistent_parent_duplicates.csv",
    index=False
)

# 4. Final curated parent-cleaned dataset
df_curated.to_csv(
    "DPP4_ChEMBL_IC50_parent_curated_unique.csv",
    index=False
)

# 5. Final QSAR-ready locked dataset
df_curated.to_csv(
    "DPP4_QSAR_dataset_parent_cleaned_locked.csv",
    index=False
)

files_to_check = [
    "DPP4_ChEMBL_IC50_strict_records_parent_annotated.csv",
    "DPP4_ChEMBL_IC50_parent_unique_before_duplicate_filter.csv",
    "DPP4_removed_inconsistent_parent_duplicates.csv",
    "DPP4_ChEMBL_IC50_parent_curated_unique.csv",
    "DPP4_QSAR_dataset_parent_cleaned_locked.csv"
]

print("Saved file check:")
for file in files_to_check:
    if os.path.exists(file):
        print(f"{file} is available")
    else:
        print(f"{file} is missing")

Saved file check:
DPP4_ChEMBL_IC50_strict_records_parent_annotated.csv is available
DPP4_ChEMBL_IC50_parent_unique_before_duplicate_filter.csv is available
DPP4_removed_inconsistent_parent_duplicates.csv is available
DPP4_ChEMBL_IC50_parent_curated_unique.csv is available
DPP4_QSAR_dataset_parent_cleaned_locked.csv is available


**Step 16. Final check-up**

In [124]:
df_qsar = pd.read_csv("DPP4_QSAR_dataset_parent_cleaned_locked.csv")

print("QSAR dataset shape:", df_qsar.shape)
print("Unique parent SMILES:", df_qsar["parent_smiles"].nunique())
print("Missing parent SMILES:", df_qsar["parent_smiles"].isna().sum())
print("Missing pIC50_median:", df_qsar["pIC50_median"].isna().sum())
print("Duplicated parent SMILES:", df_qsar["parent_smiles"].duplicated().sum())
print("Multi-fragment parent SMILES:", df_qsar["parent_smiles"].astype(str).str.contains(r"\.").sum())

print("\nActivity class distribution:")
print(df_qsar["activity_class"].value_counts())

print("\npIC50 summary:")
print(df_qsar["pIC50_median"].describe())

df_qsar.head()

QSAR dataset shape: (3666, 25)
Unique parent SMILES: 3666
Missing parent SMILES: 0
Missing pIC50_median: 0
Duplicated parent SMILES: 0
Multi-fragment parent SMILES: 0

Activity class distribution:
activity_class
High        2113
Moderate     908
Low          645
Name: count, dtype: int64

pIC50 summary:
count    3666.000000
mean        7.111067
std         1.213927
min         4.000000
25%         6.367038
50%         7.193820
75%         8.000000
max        10.096910
Name: pIC50_median, dtype: float64


,parent_smiles,molecule_chembl_id,pref_name,canonical_smiles_examples,n_records,IC50_nM_median,IC50_nM_min,IC50_nM_max,pIC50_median,pIC50_min,...,document_chembl_id,mw_freebase,alogp,hba,hbd,psa,rtb,num_ro5_violations,pIC50_range,activity_class
0,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,CHEMBL4246951,NaN,Brc1nc2n(n1)CC(c1cc3ccccc3o1)=NN2,1,28050.00,28050.00,28050.00,4.552067,4.552067,...,CHEMBL4229387,318.13,2.62,6.0,1.0,68.24,1.0,0.0,0.0,Low
1,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,CHEMBL260701,NaN,C#CCN(CC#N)C(=O)[C@@H]1C[C@H](C(=O)N(C)C)[C@H]...,1,60.00,60.00,60.00,7.221849,7.221849,...,CHEMBL1143890,276.34,-0.57,4.0,1.0,76.44,4.0,0.0,0.0,High
2,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,CHEMBL3913618,NaN,C#CCn1c(=O)n(C)c(=O)c2c1nc(N1CCCC(N)C1)n2Cc1cc...,1,16.00,16.00,16.00,7.795880,7.795880,...,CHEMBL3886676,417.47,0.38,9.0,1.0,114.87,4.0,0.0,0.0,High
3,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,CHEMBL5277490,NaN,C#CCn1c(N2CCC[C@H](N)C2)nc2c1c(=O)n(Cc1nc(C)c3...,1,1.33,1.33,1.33,8.876148,8.876148,...,CHEMBL5244277,458.53,0.76,10.0,1.0,116.86,4.0,0.0,0.0,High
4,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,CHEMBL1650422,NaN,C#Cc1cnc(N2CCC[C@@H](N)C2)n(Cc2ccccc2C#N)c1=O,1,5.00,5.00,5.00,8.301030,8.301030,...,CHEMBL1649419,333.40,1.07,6.0,1.0,87.94,3.0,0.0,0.0,High
